In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import mannwhitneyu

# ---- Adobe-friendly fonts (must be set BEFORE plotting) ----
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

base = '/Volumes/lab-windingm/home/users/cochral/LRS/AttractionRig/analysis/social-isolation/sensory'

condition_order = ['wiii8', 'anosmic', '9047', '23129', '33300']
source_order = ['real', 'pseudo']

condition_colors = {
    'wiii8': '#be0f0a',
    'anosmic': '#e7533a',
    '9047': '#fc8c59',
    '23129': '#fee2bb',
    '33300': '#fdc38d',
}
pseudo_colour = '#c4c4c4'

# Real dishes and their pseudo dishes, which hold larvae that never actually shared a plate.
real = pd.concat(
    [pd.read_csv(f'{base}/{c}/closest_contacts_1mm.csv').assign(condition=c, source='real') for c in condition_order],
    ignore_index=True,
)

pseudo = pd.concat(
    [pd.read_csv(f'{base}/{c}-pseudo/closest_contacts_1mm.csv').assign(condition=c, source='pseudo') for c in condition_order],
    ignore_index=True,
)

df = pd.concat([real, pseudo], ignore_index=True)

print(df.groupby(['condition', 'source'])['file'].nunique().unstack()[source_order].loc[condition_order])


In [ ]:
# Total frames in contact per dish: real versus its own pseudo null.
contacts_per_file = (
    df.groupby(['condition', 'source', 'file'], as_index=False)['frame']
      .size()
      .rename(columns={'size': 'total_contacts'})
)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6), sharey=True)
stats_rows = []

for ax, condition in zip(axes, condition_order):

    plot_data = contacts_per_file[contacts_per_file['condition'] == condition]
    real_values = plot_data.loc[plot_data['source'] == 'real', 'total_contacts']
    pseudo_values = plot_data.loc[plot_data['source'] == 'pseudo', 'total_contacts']

    u_stat, p_value = mannwhitneyu(real_values, pseudo_values, alternative='two-sided')
    ratio = real_values.mean() / pseudo_values.mean()
    stars = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'

    stats_rows.append({
        'condition': condition,
        'real_mean': real_values.mean(),
        'pseudo_mean': pseudo_values.mean(),
        'obs_over_exp': ratio,
        'U': u_stat,
        'p': p_value,
        'significance': stars,
    })

    sns.barplot(
        data=plot_data, x='source', y='total_contacts', order=source_order,
        hue='source', hue_order=source_order,
        palette={'real': condition_colors[condition], 'pseudo': pseudo_colour},
        errorbar='sd', legend=False, ax=ax,
    )
    sns.stripplot(
        data=plot_data, x='source', y='total_contacts', order=source_order,
        color='black', size=2.5, jitter=0.18, alpha=0.5, ax=ax,
    )

    ax.set_title(f'{condition}\nobs/exp = {ratio:.2f}  {stars}', fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Total Time in Contact Per File' if condition == condition_order[0] else '')

plt.ylim(0, None)
sns.despine()
plt.tight_layout()
plt.show()

print(pd.DataFrame(stats_rows).to_string(index=False))


In [ ]:
# Same contacts, but every dish divided by its own condition's pseudo mean, so 1 is the null.
pseudo_mean_per_condition = (
    contacts_per_file[contacts_per_file['source'] == 'pseudo']
    .groupby('condition')['total_contacts']
    .mean()
    .rename('pseudo_mean')
)

fold_contacts = contacts_per_file.merge(pseudo_mean_per_condition, on='condition')
fold_contacts['fold_change'] = fold_contacts['total_contacts'] / fold_contacts['pseudo_mean']

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6), sharey=True)
stats_rows = []

for ax, condition in zip(axes, condition_order):

    plot_data = fold_contacts[fold_contacts['condition'] == condition]
    real_values = plot_data.loc[plot_data['source'] == 'real', 'fold_change']
    pseudo_values = plot_data.loc[plot_data['source'] == 'pseudo', 'fold_change']

    u_stat, p_value = mannwhitneyu(real_values, pseudo_values, alternative='two-sided')
    stars = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'

    stats_rows.append({
        'condition': condition,
        'real_fold_change': real_values.mean(),
        'pseudo_fold_change': pseudo_values.mean(),
        'U': u_stat,
        'p': p_value,
        'significance': stars,
    })

    sns.barplot(
        data=plot_data, x='source', y='fold_change', order=source_order,
        hue='source', hue_order=source_order,
        palette={'real': condition_colors[condition], 'pseudo': pseudo_colour},
        errorbar='sd', legend=False, ax=ax,
    )
    sns.stripplot(
        data=plot_data, x='source', y='fold_change', order=source_order,
        color='black', size=2.5, jitter=0.18, alpha=0.5, ax=ax,
    )

    ax.axhline(1, color='black', linestyle='--', linewidth=1)

    ax.set_title(f'{condition}\nfold change = {real_values.mean():.2f}  {stars}', fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Contacts / Pseudo Mean' if condition == condition_order[0] else '')

plt.ylim(0, None)
sns.despine()
plt.tight_layout()
plt.show()

print(pd.DataFrame(stats_rows).to_string(index=False))


In [ ]:
real_only = df[df['source'] == 'real']

contacts_per_file_type = (
    real_only.groupby(['condition', 'file', 'Closest Interaction Type'])
    .size()
    .reset_index(name='total_contacts')
)

files = real_only[['condition', 'file']].drop_duplicates()
contact_types = real_only[['Closest Interaction Type']].drop_duplicates()
contacts_per_file_type = (
    files.merge(contact_types, how='cross')
    .merge(contacts_per_file_type, on=['condition', 'file', 'Closest Interaction Type'], how='left')
)
contacts_per_file_type['total_contacts'] = contacts_per_file_type['total_contacts'].fillna(0)
contact_type_order = contact_types['Closest Interaction Type'].tolist()

stats_rows = []

for contact_type in contact_type_order:
    control = contacts_per_file_type.loc[
        (contacts_per_file_type['condition'] == 'wiii8') &
        (contacts_per_file_type['Closest Interaction Type'] == contact_type),
        'total_contacts'
    ]

    for condition in condition_order:
        if condition == 'wiii8':
            continue

        test_group = contacts_per_file_type.loc[
            (contacts_per_file_type['condition'] == condition) &
            (contacts_per_file_type['Closest Interaction Type'] == contact_type),
            'total_contacts'
        ]

        u_stat, p_value = mannwhitneyu(test_group, control, alternative='two-sided')

        if p_value < 0.001:
            significance = '***'
        elif p_value < 0.01:
            significance = '**'
        elif p_value < 0.05:
            significance = '*'
        else:
            significance = 'ns'

        stats_rows.append({
            'contact_type': contact_type,
            'comparison': f'{condition} vs wiii8',
            'U': u_stat,
            'p': p_value,
            'significance': significance,
        })

stats_table = pd.DataFrame(stats_rows)
print(stats_table.to_string(index=False))

plt.figure(figsize=(6, 4))

ax = sns.barplot(
    data=contacts_per_file_type,
    x='Closest Interaction Type',
    y='total_contacts',
    order=contact_type_order,
    hue='condition',
    hue_order=condition_order,
    palette=condition_colors,
    errorbar='sd',
)

y_offset = contacts_per_file_type['total_contacts'].max() * 0.1
for bars, condition in zip(ax.containers, condition_order):
    if condition == 'wiii8':
        continue

    for bar, contact_type in zip(bars, contact_type_order):
        row = stats_table.loc[
            (stats_table['contact_type'] == contact_type) &
            (stats_table['comparison'] == f'{condition} vs wiii8')
        ]

        if len(row) > 0 and row['significance'].iloc[0] == '**':
            x = bar.get_x() + bar.get_width() / 2
            y = bar.get_height() + y_offset
            ax.text(x, y, '**', ha='center', va='bottom', fontsize=9)

plt.xticks(rotation=45, ha='right')
plt.ylabel('Total Contacts Per File')
plt.xlabel('Closest Interaction Type')
plt.ylim(0, None)
sns.despine()
plt.tight_layout()
plt.show()